# Quantitative Value Strategy
"Value investing" means investing in the stocks that are cheapest relative to common measures of business value (like earnings or assets).

For this project, we're going to build an investing strategy that selects the 50 stocks with the best value metrics. From there, we will calculate recommended trades for an equal-weight portfolio of these 50 stocks.

## Library Imports
The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [16]:
import numpy as np
import pandas as pd
import requests 
import xlsxwriter 
import math 
from io import StringIO
from scipy import stats 
import yfinance as yf

## Importing Our List of Stocks & API Token
As before, we'll need to import our list of stocks and our API token before proceeding. Make sure the .csv file is still in your working directory and import it with the following command:

In [17]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)

# Wrap response.text in StringIO()
# This prevents the FileNotFoundError by telling pandas: "This is a stream of text"
data_stream = StringIO(response.text)
sp500_table = pd.read_html(data_stream)

stocks = sp500_table[0]
stocks.rename(columns={'Symbol': 'Ticker'}, inplace=True)
stocks['Ticker'] = stocks['Ticker'].str.replace('.', '-', regex=False)

## Making Our First API Call
It's now time to make the first version of our value screener!

We'll start by building a simple value screener that ranks securities based on a single metric (the price-to-earnings ratio).

In [18]:
symbol = 'AAPL' 
api_url = yf.Ticker(symbol) 
data = api_url.info 
data 

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

## Parsing Our API Call
This API call has the metric we need - the price-to-earnings ratio.

Here is an example of how to parse the metric from our API call:

In [19]:
data.get('forwardPE')

31.31931

## Executing A Batch API Call & Building Our DataFrame

Just like in our first project, it's now time to execute several batch API calls and add the information we need to our DataFrame.

We'll start by running the following code cell, which contains some code we already built last time that we can re-use for this project. More specifically, it contains a function called chunks that we can use to divide our list of securities into groups of 100.

In [26]:
# # Function sourced from 
# # https://stackoverflow.com/questions/312443/how-do-you-split-a-list-into-evenly-sized-chunks
# def chunks(lst, n):
#     """Yield successive n-sized chunks from lst."""
#     for i in range(0, len(lst), n):
#         yield lst[i:i + n]   
        
# symbol_groups = list(chunks(stocks['Ticker'], 100))
# symbol_strings = []
# for i in range(0, len(symbol_groups)):
#     symbol_strings.append(','.join(symbol_groups[i]))
# #     print(symbol_strings[i])

# my_columns = ['Ticker', 'Price', 'Price-to-Earnings Ratio', 'Number of Shares to Buy']

Now we need to create a blank DataFrame and add our data to the data frame one-by-one.

In [34]:
# 1. Initialize the Tickers object with all symbols at once
# stocks['Ticker'] is your list from Wikipedia
tickers_list = stocks['Ticker'].tolist()
tickers_data = yf.Tickers(tickers_list)

rows_list = []

# print("Fetching data... this may take a minute.")

# 2. Loop through the tickers to get 'info'
# We use rows_list to avoid the slow pd.concat inside a loop
for symbol in tickers_list:
    try:
        # Accessing the individual ticker object from our batch
        ticker_info = tickers_data.tickers[symbol].info
        
        price = ticker_info.get('currentPrice') or ticker_info.get('regularMarketPrice')
        forward_pe_ratio = ticker_info.get('forwardPE') 
        
        if price:
            rows_list.append({
                'Ticker': symbol,
                'Stock Price': price,
                'Forward Price-to-Earnings Ratio': forward_pe_ratio,
                'Number of Shares to Buy': 'N/A'
            })
    except Exception as e:
        # This skips tickers that might have been delisted mid-day or have errors
        continue

# 3. Create the final DataFrame in one go
final_dataframe = pd.DataFrame(rows_list)

final_dataframe 

,Ticker,Stock Price,Forward Price-to-Earnings Ratio,Number of Shares to Buy
0,MMM,146.22,15.464474,N/A
1,AOS,56.01,13.401477,N/A
2,ABT,84.47,13.929864,N/A
3,ABBV,210.39,12.960644,N/A
4,ACN,168.82,11.324366,N/A
...,...,...,...,...
498,XYL,108.12,17.767086,N/A
499,YUM,149.97,19.988804,N/A
500,ZBRA,259.35,12.544634,N/A
501,ZBH,83.70,9.307435,N/A


## Removing Glamour Stocks

The opposite of a "value stock" is a "glamour stock". 

Since the goal of this strategy is to identify the 50 best value stocks from our universe, our next step is to remove glamour stocks from the DataFrame.

We'll sort the DataFrame by the stocks' price-to-earnings ratio, and drop all stocks outside the top 50.

In [35]:
# When you run this once, final_dataframe changes from all 500 equity names to only the top 50 names that satisfy the condition in the
# second line. Thus, if you run again with a different condition, final_dataframe will be empty because the second run is based on the 
# modified final_dataframe after the first run. 

final_dataframe.sort_values('Forward Price-to-Earnings Ratio', inplace = True)
final_dataframe = final_dataframe[final_dataframe['Forward Price-to-Earnings Ratio'] > 0]
final_dataframe = final_dataframe[:50]
final_dataframe.reset_index(inplace = True)
final_dataframe.drop('index', axis=1, inplace = True)
final_dataframe

,Ticker,Stock Price,Forward Price-to-Earnings Ratio,Number of Shares to Buy
0,CHTR,140.33,3.173662,N/A
1,GPN,67.58,4.190558,N/A
2,TPL,385.17,5.267642,N/A
3,GM,74.86,5.331669,N/A
4,OMC,70.83,5.696061,N/A
5,EG,351.67,5.797336,N/A
6,AES,14.47,6.057460,N/A
7,FIS,41.80,6.125709,N/A
8,VTRS,16.48,6.155823,N/A
9,FISV,55.33,6.176340,N/A


## Calculating the Number of Shares to Buy
We now need to calculate the number of shares we need to buy. 

To do this, we will use the `portfolio_input` function that we created in our momentum project.

I have included this function below.

In [36]:
def portfolio_input():
    global portfolio_size
    portfolio_size = input("Enter the value of your portfolio:")

    try:
        val = float(portfolio_size)
    except ValueError:
        print("That's not a number! \n Try again:")
        portfolio_size = input("Enter the value of your portfolio:")

Use the `portfolio_input` function to accept a `portfolio_size` variable from the user of this script.

In [37]:
portfolio_input()

Enter the value of your portfolio: 10000000


You can now use the global `portfolio_size` variable to calculate the number of shares that our strategy should purchase.

In [41]:
# Force the column to be numeric so it can hold the result of math.floor 
final_dataframe['Number of Shares to Buy'] = pd.to_numeric(final_dataframe['Number of Shares to Buy'], errors = 'coerce')

position_size = float(portfolio_size) / len(final_dataframe.index) 
for i in range(0, len(final_dataframe['Ticker'])): 
    final_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / final_dataframe['Stock Price'][i]) 

final_dataframe 

,Ticker,Stock Price,Forward Price-to-Earnings Ratio,Number of Shares to Buy
0,CHTR,140.33,3.173662,1425.0
1,GPN,67.58,4.190558,2959.0
2,TPL,385.17,5.267642,519.0
3,GM,74.86,5.331669,2671.0
4,OMC,70.83,5.696061,2823.0
5,EG,351.67,5.797336,568.0
6,AES,14.47,6.057460,13821.0
7,FIS,41.80,6.125709,4784.0
8,VTRS,16.48,6.155823,12135.0
9,FISV,55.33,6.176340,3614.0


## Building a Better (and More Realistic) Value Strategy
Every valuation metric has certain flaws.

For example, the price-to-earnings ratio doesn't work well with stocks with negative earnings.

Similarly, stocks that buyback their own shares are difficult to value using the price-to-book ratio.

Investors typically use a `composite` basket of valuation metrics to build robust quantitative value strategies. In this section, we will filter for stocks with the lowest percentiles on the following metrics:

* Price-to-earnings ratio
* Price-to-book ratio
* Price-to-sales ratio
* Enterprise Value divided by Earnings Before Interest, Taxes, Depreciation, and Amortization (EV/EBITDA)
* Enterprise Value divided by Gross Profit (EV/GP)

Some of these metrics aren't provided directly by the IEX Cloud API, and must be computed after pulling raw data. We'll start by calculating each data point from scratch.

In [42]:
#

Let's move on to building our DataFrame. You'll notice that I use the abbreviation `rv` often. It stands for `robust value`, which is what we'll call this sophisticated strategy moving forward.

In [66]:
rv_list = []

print("Fetching Robust Value data... this will take a few minutes.")

for symbol in tickers_list:
    try:
        # Pull the info dictionary for the specific ticker
        info = tickers_data.tickers[symbol].info
        
        # 1. Fetch Price safely
        price = info.get('currentPrice') or info.get('regularMarketPrice')
        if not price: 
            continue
            
        # 2. Extract standard multiples safely using .get()
        pe_ratio = info.get('forwardPE')
        pb_ratio = info.get('priceToBook')
        ps_ratio = info.get('priceToSalesTrailing12Months')
        ev_to_ebitda = info.get('enterpriseToEbitda')
        
        # 3. Calculate EV/GP manually (Enterprise Value / Gross Profits)
        ev = info.get('enterpriseValue')
        gp = info.get('grossProfits')
        
        if ev and gp and gp != 0:
            ev_to_gp = ev / gp
        else:
            ev_to_gp = None
            
        # 4. Append row dictionary to our list
        rv_list.append({
            'Ticker': symbol,
            'Price': price,
            'Number of Shares to Buy': 'N/A', 
            'Forward Price-to-Earnings Ratio': pe_ratio,
            'Forward PE Percentile': 'N/A',
            'Price-to-Book Ratio': pb_ratio,
            'PB Percentile': 'N/A',
            'Price-to-Sales Ratio': ps_ratio,
            'PS Percentile': 'N/A',
            'EV/EBITDA': ev_to_ebitda,
            'EV/EBITDA Percentile': 'N/A',
            'EV/GP': ev_to_gp,
            'EV/GP Percentile': 'N/A',
            'RV Score': 'N/A'
        })
    except Exception:
        # Skip any tickers that throw odd errors during parsing
        continue

# Create the clean RV DataFrame
rv_dataframe = pd.DataFrame(rv_list)
print("Data collection complete!")

rv_dataframe

Fetching Robust Value data... this will take a few minutes.
Data collection complete!


,Ticker,Price,Number of Shares to Buy,Forward Price-to-Earnings Ratio,Forward PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,MMM,146.22,N/A,15.464474,N/A,23.372763,N/A,3.047617,N/A,13.401,N/A,8.414224,N/A,N/A
1,AOS,56.01,N/A,13.401477,N/A,4.110825,N/A,2.025179,N/A,10.276,N/A,5.530745,N/A,N/A
2,ABT,84.47,N/A,13.929864,N/A,2.826124,N/A,3.259870,N/A,14.868,N/A,6.846988,N/A,N/A
3,ABBV,210.39,N/A,12.960644,N/A,-55.850810,N/A,5.917247,N/A,14.551,N/A,9.620310,N/A,N/A
4,ACN,168.82,N/A,11.324366,N/A,3.325716,N/A,1.440817,N/A,8.178,N/A,4.509875,N/A,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XYL,108.12,N/A,17.767086,N/A,2.294080,N/A,2.826908,N/A,14.231,N/A,7.751971,N/A,N/A
499,YUM,149.97,N/A,19.988804,N/A,-5.671230,N/A,4.870954,N/A,17.972,N/A,13.881268,N/A,N/A
500,ZBRA,259.35,N/A,12.544634,N/A,3.584707,N/A,2.212739,N/A,14.706,N/A,5.609190,N/A,N/A
501,ZBH,83.70,N/A,9.307435,N/A,1.277297,N/A,1.925629,N/A,9.031,N/A,3.949961,N/A,N/A


In [67]:
#

## Dealing With Missing Data in Our DataFrame

Our DataFrame contains some missing data because all of the metrics we require are not available through the API we're using. 

You can use pandas' `isnull` method to identify missing data:

In [68]:
rv_dataframe[rv_dataframe.isnull().any(axis=1)]
# rv_dataframe

,Ticker,Price,Number of Shares to Buy,Forward Price-to-Earnings Ratio,Forward PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
26,AXP,313.48,N/A,15.582090,N/A,6.288970,N/A,3.108407,N/A,NaN,N/A,5.105536,N/A,N/A
30,AMP,470.38,N/A,9.873476,N/A,6.822441,N/A,2.189159,N/A,NaN,N/A,3.410207,N/A,N/A
37,APO,135.38,N/A,12.671854,N/A,4.206700,N/A,2.494613,N/A,NaN,N/A,8.812789,N/A,N/A
58,BAC,49.77,N/A,9.870848,N/A,1.287244,N/A,3.222897,N/A,NaN,N/A,2.982109,N/A,N/A
66,BX,117.89,N/A,15.670901,N/A,11.062213,N/A,10.008201,N/A,NaN,N/A,8.228394,N/A,N/A
68,BK,135.02,N/A,14.003189,N/A,2.349073,N/A,4.465399,N/A,NaN,N/A,-3.314739,N/A,N/A
84,COF,187.17,N/A,7.752121,N/A,1.078758,N/A,3.207693,N/A,NaN,N/A,2.709111,N/A,N/A
99,SCHW,90.88,N/A,12.679862,N/A,3.719255,N/A,6.372062,N/A,NaN,N/A,5.286615,N/A,N/A
110,C,123.42,N/A,9.876373,N/A,1.099755,N/A,2.673588,N/A,NaN,N/A,0.008514,N/A,N/A
111,CFG,60.86,N/A,9.519161,N/A,1.077588,N/A,3.263994,N/A,NaN,N/A,3.424426,N/A,N/A


Dealing with missing data is an important topic in data science.

There are two main approaches:

* Drop missing data from the data set (pandas' `dropna` method is useful here)
* Replace missing data with a new value (pandas' `fillna` method is useful here)

In this tutorial, we will replace missing data with the average non-`NaN` data point from that column. 

Here is the code to do this:

In [69]:
# for column in ['Forward Price-to-Earnings Ratio', 'Price-to-Book Ratio','Price-to-Sales Ratio',  'EV/EBITDA','EV/GP']:
#     rv_dataframe[column].fillna(rv_dataframe[column].mean(), inplace = True)

# for column in ['Forward Price-to-Earnings Ratio', 'Price-to-Book Ratio','Price-to-Sales Ratio',  'EV/EBITDA','EV/GP']:
#     rv_dataframe[column].fillna({column: rv_dataframe[column].mean()}, inplace = True)



# List of columns that contain our target valuation metrics
metrics_columns = [
    'Forward Price-to-Earnings Ratio', 
    'Price-to-Book Ratio', 
    'Price-to-Sales Ratio', 
    'EV/EBITDA', 
    'EV/GP'
]

# Fill missing data with the average of each column
# for column in metrics_columns:
#     rv_dataframe[column] = pd.to_numeric(rv_dataframe[column], errors='coerce')
#     rv_dataframe[column].fillna({column: rv_dataframe[column].mean()}, inplace=True)

# Fill missing data with the average of each column (Explicit Assignment)
for column in metrics_columns:
    # 1. Force the column to be numeric
    rv_dataframe[column] = pd.to_numeric(rv_dataframe[column], errors='coerce')
    
    # 2. Calculate the mean (ignoring the NaNs)
    column_mean = rv_dataframe[column].mean()
    
    # 3. Explicitly overwrite the column with the filled version
    rv_dataframe[column] = rv_dataframe[column].fillna(column_mean)


Now, if we run the statement from earlier to print rows that contain missing data, nothing should be returned:

In [71]:
rv_dataframe[rv_dataframe.isnull().any(axis=1)]
rv_dataframe 

,Ticker,Price,Number of Shares to Buy,Forward Price-to-Earnings Ratio,Forward PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,MMM,146.22,N/A,15.464474,N/A,23.372763,N/A,3.047617,N/A,13.401,N/A,8.414224,N/A,N/A
1,AOS,56.01,N/A,13.401477,N/A,4.110825,N/A,2.025179,N/A,10.276,N/A,5.530745,N/A,N/A
2,ABT,84.47,N/A,13.929864,N/A,2.826124,N/A,3.259870,N/A,14.868,N/A,6.846988,N/A,N/A
3,ABBV,210.39,N/A,12.960644,N/A,-55.850810,N/A,5.917247,N/A,14.551,N/A,9.620310,N/A,N/A
4,ACN,168.82,N/A,11.324366,N/A,3.325716,N/A,1.440817,N/A,8.178,N/A,4.509875,N/A,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XYL,108.12,N/A,17.767086,N/A,2.294080,N/A,2.826908,N/A,14.231,N/A,7.751971,N/A,N/A
499,YUM,149.97,N/A,19.988804,N/A,-5.671230,N/A,4.870954,N/A,17.972,N/A,13.881268,N/A,N/A
500,ZBRA,259.35,N/A,12.544634,N/A,3.584707,N/A,2.212739,N/A,14.706,N/A,5.609190,N/A,N/A
501,ZBH,83.70,N/A,9.307435,N/A,1.277297,N/A,1.925629,N/A,9.031,N/A,3.949961,N/A,N/A


## Calculating Value Percentiles

We now need to calculate value score percentiles for every stock in the universe. More specifically, we need to calculate percentile scores for the following metrics for every stock:

* Price-to-earnings ratio
* Price-to-book ratio
* Price-to-sales ratio
* EV/EBITDA
* EV/GP

Here's how we'll do this:

In [75]:
rv_dataframe['Forward PE Percentile'] = pd.to_numeric(rv_dataframe['Forward PE Percentile'], errors='coerce')
rv_dataframe['PB Percentile'] = pd.to_numeric(rv_dataframe['PB Percentile'], errors='coerce')
rv_dataframe['PS Percentile'] = pd.to_numeric(rv_dataframe['PS Percentile'], errors='coerce')
rv_dataframe['EV/EBITDA Percentile'] = pd.to_numeric(rv_dataframe['EV/EBITDA Percentile'], errors='coerce')
rv_dataframe['EV/GP Percentile'] = pd.to_numeric(rv_dataframe['EV/GP Percentile'], errors='coerce')

metrics = {
            'Forward Price-to-Earnings Ratio': 'Forward PE Percentile',
            'Price-to-Book Ratio':'PB Percentile',
            'Price-to-Sales Ratio': 'PS Percentile',
            'EV/EBITDA':'EV/EBITDA Percentile',
            'EV/GP':'EV/GP Percentile'
}

for row in rv_dataframe.index:
    for metric in metrics.keys():
        rv_dataframe.loc[row, metrics[metric]] = stats.percentileofscore(rv_dataframe[metric], rv_dataframe.loc[row, metric])/100

# Print each percentile score to make sure it was calculated properly
for metric in metrics.values():
    print(rv_dataframe[metric])

#Print the entire DataFrame    
rv_dataframe

0      0.443340
1      0.341948
2      0.377734
3      0.318091
4      0.222664
         ...   
498    0.554672
499    0.646123
500    0.294235
501    0.105368
502    0.165010
Name: Forward PE Percentile, Length: 503, dtype: float64
0      0.936382
1      0.586481
2      0.445328
3      0.011928
4      0.518887
         ...   
498    0.351889
499    0.055666
500    0.538767
501    0.123260
502    0.795229
Name: PB Percentile, Length: 503, dtype: float64
0      0.469185
1      0.322068
2      0.514911
3      0.749503
4      0.204771
         ...   
498    0.439364
499    0.699801
500    0.345924
501    0.308151
502    0.518887
Name: PS Percentile, Length: 503, dtype: float64
0      0.401590
1      0.204771
2      0.473161
3      0.453280
4      0.110338
         ...   
498    0.443340
499    0.675944
500    0.459245
501    0.141153
502    0.165010
Name: EV/EBITDA Percentile, Length: 503, dtype: float64
0      0.489066
1      0.278330
2      0.385686
3      0.564612
4      0.180915
     

,Ticker,Price,Number of Shares to Buy,Forward Price-to-Earnings Ratio,Forward PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,MMM,146.22,N/A,15.464474,0.443340,23.372763,0.936382,3.047617,0.469185,13.401,0.401590,8.414224,0.489066,N/A
1,AOS,56.01,N/A,13.401477,0.341948,4.110825,0.586481,2.025179,0.322068,10.276,0.204771,5.530745,0.278330,N/A
2,ABT,84.47,N/A,13.929864,0.377734,2.826124,0.445328,3.259870,0.514911,14.868,0.473161,6.846988,0.385686,N/A
3,ABBV,210.39,N/A,12.960644,0.318091,-55.850810,0.011928,5.917247,0.749503,14.551,0.453280,9.620310,0.564612,N/A
4,ACN,168.82,N/A,11.324366,0.222664,3.325716,0.518887,1.440817,0.204771,8.178,0.110338,4.509875,0.180915,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XYL,108.12,N/A,17.767086,0.554672,2.294080,0.351889,2.826908,0.439364,14.231,0.443340,7.751971,0.439364,N/A
499,YUM,149.97,N/A,19.988804,0.646123,-5.671230,0.055666,4.870954,0.699801,17.972,0.675944,13.881268,0.751491,N/A
500,ZBRA,259.35,N/A,12.544634,0.294235,3.584707,0.538767,2.212739,0.345924,14.706,0.459245,5.609190,0.290258,N/A
501,ZBH,83.70,N/A,9.307435,0.105368,1.277297,0.123260,1.925629,0.308151,9.031,0.141153,3.949961,0.141153,N/A


## Calculating the RV Score
We'll now calculate our RV Score (which stands for Robust Value), which is the value score that we'll use to filter for stocks in this investing strategy.

The RV Score will be the arithmetic mean of the 4 percentile scores that we calculated in the last section.

To calculate arithmetic mean, we will use the mean function from Python's built-in statistics module.

In [77]:
rv_dataframe['RV Score'] = pd.to_numeric(rv_dataframe['RV Score'], errors='coerce')

from statistics import mean

for row in rv_dataframe.index:
    value_percentiles = []
    for metric in metrics.keys():
        value_percentiles.append(rv_dataframe.loc[row, metrics[metric]])
    rv_dataframe.loc[row, 'RV Score'] = mean(value_percentiles)
    
rv_dataframe

,Ticker,Price,Number of Shares to Buy,Forward Price-to-Earnings Ratio,Forward PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,MMM,146.22,N/A,15.464474,0.443340,23.372763,0.936382,3.047617,0.469185,13.401,0.401590,8.414224,0.489066,0.547913
1,AOS,56.01,N/A,13.401477,0.341948,4.110825,0.586481,2.025179,0.322068,10.276,0.204771,5.530745,0.278330,0.346720
2,ABT,84.47,N/A,13.929864,0.377734,2.826124,0.445328,3.259870,0.514911,14.868,0.473161,6.846988,0.385686,0.439364
3,ABBV,210.39,N/A,12.960644,0.318091,-55.850810,0.011928,5.917247,0.749503,14.551,0.453280,9.620310,0.564612,0.419483
4,ACN,168.82,N/A,11.324366,0.222664,3.325716,0.518887,1.440817,0.204771,8.178,0.110338,4.509875,0.180915,0.247515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XYL,108.12,N/A,17.767086,0.554672,2.294080,0.351889,2.826908,0.439364,14.231,0.443340,7.751971,0.439364,0.445726
499,YUM,149.97,N/A,19.988804,0.646123,-5.671230,0.055666,4.870954,0.699801,17.972,0.675944,13.881268,0.751491,0.565805
500,ZBRA,259.35,N/A,12.544634,0.294235,3.584707,0.538767,2.212739,0.345924,14.706,0.459245,5.609190,0.290258,0.385686
501,ZBH,83.70,N/A,9.307435,0.105368,1.277297,0.123260,1.925629,0.308151,9.031,0.141153,3.949961,0.141153,0.163817


## Selecting the 50 Best Value Stocks¶

As before, we can identify the 50 best value stocks in our universe by sorting the DataFrame on the RV Score column and dropping all but the top 50 entries.

In [78]:
rv_dataframe.sort_values(by = 'RV Score', inplace = True)
rv_dataframe = rv_dataframe[:50]
rv_dataframe.reset_index(drop = True, inplace = True)

## Calculating the Number of Shares to Buy
We'll use the `portfolio_input` function that we created earlier to accept our portfolio size. Then we will use similar logic in a for loop to calculate the number of shares to buy for each stock in our investment universe.

In [106]:
portfolio_input()

Enter the value of your portfolio: 10000000


In [107]:
rv_dataframe['Number of Shares to Buy'] = pd.to_numeric(rv_dataframe['Number of Shares to Buy'], errors='coerce')

position_size = float(portfolio_size) / len(rv_dataframe.index)
for i in range(0, len(rv_dataframe['Ticker'])-1):
    rv_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / rv_dataframe['Price'][i])
rv_dataframe

,Ticker,Price,Number of Shares to Buy,Forward Price-to-Earnings Ratio,Forward PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,HPQ,20.81,9610.0,6.996392,0.039761,-24.922155,0.025845,0.339740,0.023857,6.017,0.043738,2.354948,0.039761,0.034592
1,CMCSA,24.76,8077.0,6.494853,0.027833,1.003282,0.085487,0.706022,0.087475,4.915,0.017893,1.978879,0.027833,0.049304
2,CHTR,140.33,1425.0,3.173662,0.007952,1.053307,0.089463,0.362610,0.025845,5.381,0.021869,3.910461,0.131213,0.055268
3,UHS,168.64,1185.0,6.635895,0.033797,1.367588,0.147117,0.574813,0.067594,5.756,0.035785,1.954612,0.025845,0.062028
4,TAP,40.84,4897.0,8.143959,0.067594,0.761685,0.071571,0.684638,0.085487,5.720,0.033797,3.236571,0.083499,0.068390
5,EPAM,93.02,2150.0,6.594123,0.029821,1.430285,0.163022,0.874781,0.113320,5.621,0.029821,2.547071,0.043738,0.075944
6,UAL,92.85,2154.0,6.596236,0.031809,1.911280,0.274354,0.498409,0.039761,5.953,0.041750,2.297166,0.037773,0.085089
7,LEN,82.30,2430.0,10.843329,0.192843,0.926458,0.079523,0.611015,0.071571,6.829,0.061630,3.029257,0.065606,0.094235
8,CTSH,47.13,4243.0,7.650763,0.051690,1.482588,0.176938,1.041413,0.143141,5.631,0.031809,3.053871,0.067594,0.094235
9,APTV,54.34,3680.0,8.011051,0.059642,1.249770,0.121272,0.556632,0.059642,5.809,0.039761,4.733042,0.192843,0.094632


## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

In [108]:
writer = pd.ExcelWriter('value_strategy.xlsx', engine='xlsxwriter')
rv_dataframe.to_excel(writer, sheet_name='Value Strategy', index = False)

## Creating the Formats We'll Need For Our .xlsx File
You'll recall from our first project that formats include colors, fonts, and also symbols like % and $. We'll need four main formats for our Excel document:

* String format for tickers
* \$XX.XX format for stock prices
* \$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase
* Float formats with 1 decimal for each valuation metric

Since we already built some formats in past sections of this course, I've included them below for you. Run this code cell before proceeding.

In [109]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_template = writer.book.add_format(
        {
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

dollar_template = writer.book.add_format(
        {
            'num_format':'$0.00',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

integer_template = writer.book.add_format(
        {
            'num_format':'0.00',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

float_template = writer.book.add_format(
        {
            'num_format':'0.0000',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

percent_template = writer.book.add_format(
        {
            'num_format':'0.0%',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

In [110]:
column_formats = {
                    'A': ['Ticker', string_template],
                    'B': ['Price', dollar_template],
                    'C': ['Number of Shares to Buy', integer_template],
                    'D': ['Forward Price-to-Earnings Ratio', float_template],
                    'E': ['Forward PE Percentile', percent_template],
                    'F': ['Price-to-Book Ratio', float_template],
                    'G': ['PB Percentile',percent_template],
                    'H': ['Price-to-Sales Ratio', float_template],
                    'I': ['PS Percentile', percent_template],
                    'J': ['EV/EBITDA', float_template],
                    'K': ['EV/EBITDA Percentile', percent_template],
                    'L': ['EV/GP', float_template],
                    'M': ['EV/GP Percentile', percent_template],
                    'N': ['RV Score', float_template]
                 }

for column in column_formats.keys():
    writer.sheets['Value Strategy'].set_column(f'{column}:{column}', 25, column_formats[column][1])
    writer.sheets['Value Strategy'].write(f'{column}1', column_formats[column][0], column_formats[column][1])

## Saving Our Excel Output
As before, saving our Excel output is very easy:

In [111]:
writer.close()